### 0) Imports

In [1]:
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import CountEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

### 1) Download data from Don’tGetKicked competition

In [2]:
df = pd.read_csv('../datasets/training.csv')

### 2) Time split

In [3]:
def split_test_valid_train(data, date_col):
    df = data.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df.sort_values(by=date_col, ignore_index=True, inplace=True)
    
    uniq_dates = sorted(df[date_col].unique())
    n = len(uniq_dates)
    train_end = uniq_dates[n // 3]
    valid_end = uniq_dates[2 * n // 3]
    
    train = df[df[date_col] < train_end].copy()
    valid = df[(df[date_col] >= train_end) & (df[date_col] < valid_end)].copy()
    test  = df[df[date_col] >= valid_end].copy()
    
    return train, valid, test

In [4]:
X_train_orig, X_valid_orig, X_test_orig = split_test_valid_train(df, 'PurchDate')

### 3) Preprocessing categorical variables

In [5]:
df.head(10)

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
5,6,0,12/7/2009,ADESA,2004,5,MITSUBISHI,GALANT 4C,ES,4D SEDAN ES,...,8149.0,9451.0,NaN,NaN,19638,33619,FL,5600.0,0,594
6,7,0,12/7/2009,ADESA,2004,5,KIA,SPECTRA,EX,4D SEDAN EX,...,6230.0,8603.0,NaN,NaN,19638,33619,FL,4200.0,0,533
7,8,0,12/7/2009,ADESA,2005,4,FORD,TAURUS,SE,4D SEDAN SE,...,6942.0,8242.0,NaN,NaN,19638,33619,FL,4500.0,0,825
8,9,0,12/7/2009,ADESA,2007,2,KIA,SPECTRA,EX,4D SEDAN EX,...,9637.0,10778.0,NaN,NaN,21973,33619,FL,5600.0,0,482
9,10,0,12/7/2009,ADESA,2007,2,FORD,FIVE HUNDRED,SEL,4D SEDAN SEL,...,12580.0,14845.0,NaN,NaN,21973,33619,FL,7700.0,0,1633


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [7]:
y_train, y_valid, y_test = X_train_orig['IsBadBuy'], X_valid_orig['IsBadBuy'], X_test_orig['IsBadBuy']

In [8]:
drop_cols = ['IsBadBuy', 'PurchDate', 'RefID', 'KickDate', 'BYRNO', 'WheelTypeID']
ohe_cols = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART']
count_cols = ['Make', 'Model', 'Trim', 'SubModel', 'Color', 'VNST']
num_cols = df.select_dtypes('number').columns.tolist()
num_cols = [num for num in num_cols if num not in drop_cols]

In [9]:
imputer = SimpleImputer(strategy='median')

In [10]:
ohe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [11]:
count = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('count', CountEncoder(handle_unknown=0))])

In [12]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', imputer, num_cols),
        ('ohe', ohe, ohe_cols),
        ('count', count, count_cols),
    ],
    remainder='drop'
)

In [13]:
X_train = preprocess.fit_transform(X_train_orig)
X_valid = preprocess.transform(X_valid_orig)
X_test = preprocess.transform(X_test_orig)

### 4) Train LogisticRegression, GaussianNB, KNN

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

In [15]:
def Gini(y_true, y_pred_proba):
    return np.abs(2 * roc_auc_score(y_true, y_pred_proba) - 1)

In [16]:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
y_pred_proba = logreg.predict_proba(X_valid)[:, 1]

In [17]:
print(f'Gini score for LogisticRegression = {Gini(y_valid, y_pred_proba)}')

Gini score for LogisticRegression = 0.3510889531208503


In [18]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_proba = gnb.predict_proba(X_valid)[:, 1]

In [19]:
print(f'Gini score for GaussianNB = {Gini(y_valid, y_pred_proba)}')

Gini score for GaussianNB = 0.2950407734435967


In [20]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred_proba = knn.predict_proba(X_valid)[:, 1]

In [21]:
print(f'Gini score for KNN = {Gini(y_valid, y_pred_proba)}')

Gini score for KNN = 0.15945429849530912


Лучшая модель - это логистическая регрессия, потому что данная модель хорошо улавливает линейные признаки или монотонные, например, если машина уже старанная или у нее большой пробег, то очевидно, что машина хуже, чем какие то другие. Также были применены OneHotEncoder и StandardScaler, что также улучшает сходимость модели

### 5) Implement Gini score calculation

In [22]:
def my_Gini(y_true, y_pred_proba):
    pos = y_pred_proba[y_true == 1]
    neg = y_pred_proba[y_true == 0]

    n_pos = len(pos)
    n_neg = len(neg)

    combined = np.concatenate([pos, neg])
    ranks = np.argsort(np.argsort(combined)) + 1
    pos_ranks = np.sum(ranks[:n_pos])
    
    my_roc_auc_score = (pos_ranks - (n_pos * (n_pos + 1)) / 2) / (n_pos * n_neg)
    return np.abs(2 * my_roc_auc_score - 1)

In [23]:
logreg.fit(X_train, y_train)
y_pred_proba = logreg.predict_proba(X_valid)[:, 1]

In [24]:
print(f'Gini score for LogisticRegression = {my_Gini(y_valid, y_pred_proba)}')

Gini score for LogisticRegression = 0.35108895312085053


### 6) Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers

$$
L_{log} \left( y, \hat{p} \right) = -logPr(y|\hat{p}) = -\frac{1}{n}\sum_{i=0}^{n-1} \left( y_i\ln(p_i) + (1 - y_i)\ln(1-p_i) \right)
$$

$$
\frac{\partial{L}}{\partial{w_k}} = \frac{\partial{L}}{\partial{p}} \frac{\partial{p}}{\partial{a}} \frac{\partial{a}}{\partial{w_k}}
$$

$$
\frac{\partial{L}}{\partial{p}} = -\frac{1}{n}\sum_{i=0}^{n-1} \left( \frac{y_i}{p_i} - \frac{1-y_i}{1-p_i} \right)
$$

$$
\frac{\partial{p}}{\partial{a}} = \frac{e^{-a}}{(1+e^{-a})^2} = \frac{1}{1+e^{-a}} \frac{e^{-a}}{1+e^{-a}} = \frac{1}{1+e^{-a}} \left( 1 - \frac{1}{1+e^{-a}} \right) = p_i(1-p_i)
$$

$$
\frac{\partial{a}}{\partial{w_k}} = x_k, k=0, ..., m+1
$$

$$
\frac{\partial{L}}{\partial{w_k}} = \frac{1}{n}\sum_{i=0}^{n-1}(p_i - y_i)x_{i,k}, k=0, ..., m+1
$$

In [25]:
class MyLogisticRegression:
    def __init__(self, threshold=0.5, learning_rate=0.001, epochs=100, batch_size=128, seed=21):
        self.threshold = threshold
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed
        self.weights = None
        self.bias = None

    def sigmoid_(self, a):
        return 1 / (1 + np.exp(-a))

    def fit(self, X, y):
        np.random.seed(self.seed)
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)

        n_samples, m_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)

        self.weights = np.zeros((m_features+1, 1))

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            for i in range(0, n_samples, self.batch_size):
                batch = indices[i:i+self.batch_size]
                X_batch = X[batch]
                y_batch = y[batch]
                a = X_batch @ self.weights
                p = self.sigmoid_(a)

                gradient = 1 / (X_batch.shape[0]) * X_batch.T @ (p - y_batch)
                self.weights -= self.lr * gradient

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

    def predict_proba(self, X):
        X = np.array(X)
        a = X @ self.weights + self.bias
        return self.sigmoid_(a).reshape(-1)

    def predict(self, X):
        X = np.array(X)
        return (self.predict_proba(X) >= self.threshold).astype(int)

In [26]:
mylogreg = MyLogisticRegression(batch_size=2)
mylogreg.fit(X_train, y_train)
y_pred_proba = mylogreg.predict_proba(X_valid)

In [27]:
print(f'Gini score for LogisticRegression = {Gini(y_valid, y_pred_proba)}')

Gini score for LogisticRegression = 0.34843785735476596


$$
f(x) = \frac{1}{\sqrt{2\pi}\sigma}e^{-\frac{1}{2} \left( \frac{x-\mu}{\sigma} \right)^2}
$$

$$
\ln(f(x)) = - \frac{1}{2}\ln(2\pi\sigma^2) - \frac{(x-\mu)^2}{2\sigma^2}
$$

$$
P(A|B) = \frac{P(B|A)P(A)}{P(B)}
$$

In [28]:
class MyGaussianNB:
    def __init__(self, var_smoothing=1e-09):
        self.vs = var_smoothing

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1)

        n_samples, m_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = self.classes_.shape[0]

        self.theta_ = np.zeros((n_classes, m_features))
        self.var_ = np.zeros((n_classes, m_features))
        self.class_prior_ = np.zeros(n_classes)

        epsilon = self.vs * np.max(np.var(X, axis=0))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.theta_[i, :] = np.mean(X_c, axis=0)
            self.var_[i, :] = np.var(X_c, axis=0) + epsilon
            self.class_prior_[i] = X_c.shape[0] / n_samples

    def _joint_log_likelihood(self, X):
        X = np.array(X)
        joint_log_likelihood = []
        for i in range(len(self.classes_)):
            prior = np.log(self.class_prior_[i])
            mean = self.theta_[i]
            var = self.var_[i]
            norm_dist = -0.5 * np.sum(np.log(2. * np.pi * var))
            norm_dist -= 0.5 * np.sum(((X - mean) ** 2) / var, axis=1)
            joint_log_likelihood.append(prior + norm_dist)
        return np.array(joint_log_likelihood).T

    def predict_proba(self, X):
        X = np.array(X)
        jll = self._joint_log_likelihood(X)
        log_prob_x = np.logaddexp.reduce(jll, axis=1, keepdims=True)
        probs = np.exp(jll - log_prob_x)
        return probs

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

In [29]:
mygnb = MyGaussianNB()
mygnb.fit(X_train, y_train)
y_pred_proba = mygnb.predict_proba(X_valid)[:,1]

In [30]:
print(f'Gini score for GaussianNB = {Gini(y_valid, y_pred_proba)}')

Gini score for GaussianNB = 0.2950407734435967


In [31]:
class MyKNeighborsClassifier:
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y).reshape(-1, 1)

    def _compute_distances(self, X):
        dists = np.sqrt(np.sum(X**2, axis=1).reshape(-1, 1) - 2 * X @ self.X_train.T + np.sum(self.X_train**2, axis=1))
        return dists

    def predict_proba(self, X):
        X = np.array(X)
        dists = self._compute_distances(X)
        neighbors_indices = np.argpartition(dists, self.n_neighbors, axis=1)[:, :self.n_neighbors]
        neighbors_labels = self.y_train[neighbors_indices]
        probabilities = np.mean(neighbors_labels == 1, axis=1)
        
        return probabilities

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

In [32]:
myknn = MyKNeighborsClassifier()
myknn.fit(X_train, y_train)
y_pred_proba = myknn.predict_proba(X_valid)

In [33]:
print(f'Gini score for KNN = {Gini(y_valid, y_pred_proba)}')

Gini score for KNN = 0.15945429849530912


### 7) Try to create non-linear features

In [34]:
X_train_orig['Auction_Avg_Price'] = X_train_orig['Auction'].map(X_train_orig.groupby('Auction')['VehBCost'].mean())
X_valid_orig['Auction_Avg_Price'] = X_valid_orig['Auction'].map(X_train_orig.groupby('Auction')['VehBCost'].mean())
X_test_orig['Auction_Avg_Price'] = X_test_orig['Auction'].map(X_train_orig.groupby('Auction')['VehBCost'].mean())

In [35]:
num_cols.append('Auction_Avg_Price')

In [36]:
X_train = preprocess.fit_transform(X_train_orig)
X_valid = preprocess.transform(X_valid_orig)
X_test = preprocess.transform(X_test_orig)

In [37]:
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

In [38]:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
y_pred_proba = logreg.predict_proba(X_valid)[:, 1]

In [39]:
print(f'Gini score for LogisticRegression = {Gini(y_valid, y_pred_proba)}')

Gini score for LogisticRegression = 0.35112286109577173


In [40]:
gnb.fit(X_train, y_train)
y_pred_proba = gnb.predict_proba(X_valid)[:, 1]

In [41]:
print(f'Gini score for GaussianNB = {Gini(y_valid, y_pred_proba)}')

Gini score for GaussianNB = 0.29407605421350436


In [42]:
knn.fit(X_train, y_train)
y_pred_proba = knn.predict_proba(X_valid)[:, 1]

In [43]:
print(f'Gini score for KNN = {Gini(y_valid, y_pred_proba)}')

Gini score for KNN = 0.16244081433930324


### 8) Determine the best features for the problem using the coefficients of the logistic model

In [44]:
logreg_l1 = LogisticRegression(solver='liblinear', l1_ratio=1.0,  C=0.01)
logreg_l1.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",1.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mu

In [45]:
ohe_names = preprocess.named_transformers_['ohe']['encoder'].get_feature_names_out(ohe_cols).tolist()
all_feature_names = num_cols + ohe_names + count_cols

In [46]:
features = pd.DataFrame({'Name': all_feature_names,'Importance': np.abs(logreg_l1.coef_[0])})
features.sort_values(by='Importance', inplace=True, ascending=False)
features

,Name,Importance
13,VehBCost,0.224388
1,VehYear,0.194060
23,WheelType_Covers,0.136265
3,VehOdo,0.130916
48,Model,0.112970
22,WheelType_Alloy,0.089197
2,VehicleAge,0.066922
17,Auction_ADESA,0.065114
0,RefId,0.054443
12,VNZIP1,0.049988


### 9) Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset

In [47]:
def objective(trial):
    c_val = trial.suggest_float('C', 1e-4, 100, log=True)
    model = LogisticRegression(solver='liblinear', l1_ratio=1.0,  C=c_val)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_valid)[:,1]
    return Gini(y_valid, y_pred_proba)

In [48]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-01-04 23:29:16,009] A new study created in memory with name: no-name-86bf47bb-8280-44df-8641-b56b63ac09f9
[I 2026-01-04 23:29:17,019] Trial 0 finished with value: 0.3414941015882631 and parameters: {'C': 0.5148242007352991}. Best is trial 0 with value: 0.3414941015882631.
[I 2026-01-04 23:29:27,486] Trial 1 finished with value: 0.19739524612880333 and parameters: {'C': 2.5022824112231037}. Best is trial 0 with value: 0.3414941015882631.
[I 2026-01-04 23:29:27,521] Trial 2 finished with value: 0.0 and parameters: {'C': 0.0005032044884390876}. Best is trial 0 with value: 0.3414941015882631.
[I 2026-01-04 23:29:32,348] Trial 3 finished with value: 0.33794625191626726 and parameters: {'C': 0.11359333795369281}. Best is trial 0 with value: 0.3414941015882631.
[I 2026-01-04 23:29:48,724] Trial 4 finished with value: 0.15036115055465005 and parameters: {'C': 4.854499337018731}. Best is trial 0 with value: 0.3414941015882631.
[I 2026-01-04 23:29:51,724] Trial 5 finished with value: 0.3

In [49]:
print(study.best_params)

{'C': 0.9656608411695037}


Для логистической регрессии гиперпараметр, который сильнее всего влияет на модель, это регуляризация

### 10) Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini

In [50]:
result_Gini = pd.DataFrame(columns=['model', 'train', 'valid', 'test'])

In [51]:
logreg = LogisticRegression(solver='liblinear', l1_ratio=1.0,  C=study.best_params['C'])
logreg.fit(X_train, y_train)
y_pred_proba_train = logreg.predict_proba(X_train)[:,1]
y_pred_proba_valid = logreg.predict_proba(X_valid)[:,1]
y_pred_proba_test = logreg.predict_proba(X_test)[:,1]

In [52]:
result_Gini.loc[len(result_Gini)] = ['Logistic Regression', Gini(y_train, y_pred_proba_train), Gini(y_valid, y_pred_proba_valid), Gini(y_test, y_pred_proba_test)]

In [53]:
result_Gini

,model,train,valid,test
0,Logistic Regression,0.388322,0.34342,0.352787


Модель не переобучилась, небольшая разница между тренировочными данными и тестовыми понятна, потому что модель обучалась на тренировочных, а валидационные данные дают даже хуже результат, чем тестовые, что показывает, что модель стабильна и не подстроилась случайно под валидационный набор данных

### 11) Implement calculation of Recall, Precision, F1 score and AUC PR metrics

In [54]:
def MyRecall(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn)

In [55]:
def MyPrecision(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tp / (tp + fp)

In [56]:
def MyF1_score(y_true, y_pred):
    prec = MyPrecision(y_true, y_pred)
    rec = MyRecall(y_true, y_pred)
    return 2 * prec * rec / (prec + rec)

In [60]:
def MyAveragePrecision(y_true, y_pred_prob):
    indices = np.argsort(y_pred_prob)[::-1]
    y_true = np.array(y_true)[indices]
    
    tp = 0
    fp = 0
    total_positives = np.sum(y_true)
    ap = 0
    prev_recall = 0
    
    for i in range(len(y_true)):
        if y_true[i] == 1:
            tp += 1
        else:
            fp += 1
        
        current_precision = tp / (tp + fp)
        current_recall = tp / total_positives
        
        if current_recall > prev_recall:
            ap += current_precision * (current_recall - prev_recall)
            prev_recall = current_recall
            
    return ap

In [58]:
mylogreg.fit(X_train, y_train)
y_pred = mylogreg.predict_proba(X_test)

In [61]:
print(MyAveragePrecision(y_test, y_pred))

0.23134484012192466


In [62]:
mygnb.fit(X_train, y_train)
y_pred_proba = mygnb.predict_proba(X_test)

In [63]:
print(MyAveragePrecision(y_test, y_pred))

0.23134484012192466


In [64]:
myknn.fit(X_train, y_train)
y_pred_proba = myknn.predict_proba(X_test)

In [65]:
print(MyAveragePrecision(y_test, y_pred))

0.23134484012192466


### 12) Which hard label metric do you prefer for the task of detecting "lemon" cars?

В данном контексте я выберу F1 score, т.к. это баланс между эффективность, которую дает Precision, и безопасностью, которую дает Recall